In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Evaluation of Formulas from First Order Logic

In this notebook we show how formulas from *first order logic* can be evaluated in F#.

## The Axioms of Group Theory

To have a nontrivial example of formulas, we use the formulas from
[group theory](https://en.wikipedia.org/wiki/Group_theory).
A [group](https://en.wikipedia.org/wiki/Group_(mathematics)) is defined as a triple
$$ \langle G, \mathrm{e}, \circ \rangle $$
where
- $G$ is a non-empty set,
- $\mathrm{e}$ is an element from $G$, and
- $\circ:G \times G \rightarrow G$ is a binary function on $G$.
- Furthermore, the following axioms have to be satisfied:
  * $\forall x: \mathrm{e} \circ x = x$,
  * $\forall x: \exists{y}: y \circ x = \mathrm{e}$,
  * $\forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z)$.
- A group is <em style="color:blue">commutative</em> if, additionally, the following formula is satisfied:
  $$\forall x: \forall y: x \circ y = y \circ x. $$

The notebook `FOL-Parser.ipynb` implements a parser for first order logic.  It also defines the types `Term` and `Formula`:
```
type Term =
    | Var of string                  // variable
    | Fun of string * Term list      // function application, constants have no arguments

type Formula =
    | Verum                          // ⊤
    | Falsum                         // ⊥
    | Atom   of string * Term list   // atomic formula P(t1, ..., tn)
    | Not    of Formula              // ¬
    | And    of Formula * Formula    // ∧
    | Or     of Formula * Formula    // ∨
    | Imp    of Formula * Formula    // →
    | Iff    of Formula * Formula    // ↔
    | Xor    of Formula * Formula    // ⊕
    | Forall of string * Formula     // ∀x: f
    | Exists of string * Formula     // ∃x: f
```

In [2]:
importNotebook "FOL-Parser.ipynb"

imported 

FOL-Parser.ipynb

The parser distinguishes between variables and function symbols as follows:
- A word starting with a lower case letter is interpreted as a *variable*.
- A word starting with an upper case letter is assumed to be a *function* or
  *predicate symbol*.

Therefore, we represent the symbols from group theory as follows:
- The neutral element $\mathrm{e}$ of group theory is represented as the nullary function symbol `E`.
- As our parser does not support using the symbol $\circ$ as a binary operator, we will use the function symbol
  `Multiply` to represent this operator.
- The predicate symbol $=$ is represented as `Equals`.

Then the formulas of group theory can be represented as follows:

In [3]:
let G1 = "∀x:Equals(Multiply(E(),x),x)"

In [4]:
let G2 = "∀x:∃y:Equals(Multiply(y,x),E())"

In [5]:
let G3 = "∀x:∀y:∀z:Equals(Multiply(Multiply(x,y),z), Multiply(x,Multiply(y,z)))"

In [6]:
let G4 = "∀x:∀y:Equals(Multiply(x,y), Multiply(y,x))"

The function `parse` from the notebook `FOL-Parser.ipynb` takes a string and converts it into a formula.

In [7]:
let F1 = parse G1
F1

Forall ("x", Atom ("Equals", [Fun ("Multiply", [Fun ("E", []); Var "x"]); Var "x"]))

In [8]:
let F2 = parse G2
F2

Forall ("x", Exists ("y", Atom ("Equals", [Fun ("Multiply", [Var "y"; Var "x"]); Fun ("E", [])])))

In [9]:
let F3 = parse G3
F3

Forall ("x", Forall ("y", Forall ("z", Atom ("Equals", [Fun ("Multiply", [Fun ("Multiply", [Var "x"; Var "y"]); Var "z"]); Fun ("Multiply", [Var "x"; Fun ("Multiply", [Var "y"; Var "z"])])]))))

In [10]:
let F4 = parse G4
F4

Forall ("x", Forall ("y", Atom ("Equals", [Fun ("Multiply", [Var "x"; Var "y"]); Fun ("Multiply", [Var "y"; Var "x"])])))

## A Structure for Group Theory

A *first order structure* $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ consists of a universe $\mathcal{U}$ and an interpretation $\mathcal{J}$ of the function and predicate symbols.  In F#, we represent a structure as a *record*.  A record is a type with named fields.  The type `Structure` has three fields:
- `Universe` is the set $\mathcal{U}$.  We use integers as the elements of the universe.
- `Functions` is a map that assigns to every function symbol $f$ its interpretation $f^\mathcal{J}$.  The interpretation of an $n$-ary function symbol is itself a map that assigns a value to every list of $n$ arguments.
- `Predicates` is a map that assigns to every predicate symbol $p$ its interpretation $p^\mathcal{J}$.  The interpretation of an $n$-ary predicate symbol is the set of those lists of $n$ elements for which the predicate is true.

In Python, both kinds of interpretations are stored in the same dictionary.  As functions and predicates are interpreted by values of different types, we need two different maps in F#.

In [11]:
type Structure =
    { Universe   : Set<int>
      Functions  : Map<string, Map<int list, int>>
      Predicates : Map<string, Set<int list>> }

The smallest non-trivial group has just two elements.  Therefore, we can define the universe `U` as follows:

In [12]:
let U = set [ 0; 1 ]

Next, we need to define the function that interprets the nullary function symbol `E`.  We define this function as a map that maps the empty list to the element `0`.

In [13]:
let NeutralElement : Map<int list, int> = Map [ ([], 0) ]

The binary function symbol `Multiply` is interpreted by the map `Product`:

In [14]:
let Product = Map [ ([0; 0], 0); ([0; 1], 1); ([1; 0], 1); ([1; 1], 0) ]

The predicate symbol `Equals` is interpreted by the binary relation `Identity`.

In [15]:
let Identity = set [ for x in U -> [x; x] ]
Identity

set [[0; 0]; [1; 1]]

Now we can define the *first order structure* $\mathcal{S}$.  Values of a record type are written in curly braces.

In [16]:
let S = { Universe   = U
          Functions  = Map [ ("E", NeutralElement); ("Multiply", Product) ]
          Predicates = Map [ ("Equals", Identity) ] }
S

{ Universe = set [0; 1]; Functions = map [("E", map [([], 0)]); ("Multiply", map [([0; 0], 0); ([0; 1], 1); ([1; 0], 1); ([1; 1], 0)])]; Predicates = map [("Equals", set [[0; 0]; [1; 1]])] }

Finally, we define the *variable assignment* $\mathcal{I}$ for the variables $x$, $y$, and $z$.  A variable assignment is a map from variable names to elements of the universe.

In [17]:
let I = Map [ ("x", 0); ("y", 1); ("z", 0) ]
I

map [("x", 0); ("y", 1); ("z", 0)]

## Functions to Evaluate Formulas

The function `evalTerm` takes three arguments:
- `t` is a term,
- `S` is a first order structure,
- `I` is a variable assignment.

It returns the value of the term `t` in the structure `S` using the variable assignment `I`.

**Implementation:** If `t` is a variable, its value is given by `I`.  Otherwise, `t` has the form $f(t_1, \cdots, t_n)$.  Then the arguments $t_1$, $\cdots$, $t_n$ are evaluated recursively and the interpretation of $f$ is applied to the list of their values.

In [18]:
let rec evalTerm (t: Term) (S: Structure) (I: Map<string, int>) : int =
    match t with
    | Var x        -> I[x]
    | Fun (f, args) ->
        let fJ      = S.Functions[f]                          // interpretation of function symbol
        let ArgVals = [ for arg in args -> evalTerm arg S I ] // recursively evaluate arguments
        fJ[ArgVals]

In [19]:
let t = parseTerm "Multiply(E(),x)"
t

Fun ("Multiply", [Fun ("E", []); Var "x"])

In [20]:
evalTerm t S I

0

The function `evalAtomic` takes three arguments:
- `p` is a predicate symbol,
- `Args` is the list of arguments of `p`,
- `S` is a first order structure,
- `I` is a variable assignment.

It returns the truth value of the atomic formula $p(t_1, \cdots, t_n)$ in the structure `S` using the variable assignment `I`.  This formula is true if the list of the values of the arguments is an element of the interpretation of `p`.

In [21]:
let evalAtomic (p: string) (Args: Term list) (S: Structure) (I: Map<string, int>) : bool =
    let pJ      = S.Predicates[p]                         // interpretation of predicate symbol
    let ArgVals = [ for arg in Args -> evalTerm arg S I ]
    pJ.Contains ArgVals

In [22]:
let f = parse "Equals(Multiply(E(),x),x)"
f

Atom ("Equals", [Fun ("Multiply", [Fun ("E", []); Var "x"]); Var "x"])

In [23]:
match f with
| Atom (p, args) -> evalAtomic p args S I
| _              -> failwith "not an atomic formula"

true

Given a variable assignment $\mathcal{I}$, a variable $x$, and an element $c$ from the universe $\mathcal{U}$, the function $\texttt{modify}(\mathcal{I}, x, c)$ computes the variable assignment $\mathcal{I}[x/c]$ which is defined for all variables $y$ as follows:
$$ I[x/c](y) = \left\{ \begin{array}{ll}
                        c     & \mbox{if $x = y$,}  \\
                        I(y)  & \mbox{otherwise.}
                        \end{array}
               \right.
$$
As F# maps are immutable, `I.Add(x, c)` returns a new map and leaves `I` unchanged.

In [24]:
let modify (I: Map<string, int>) (x: string) (c: int) : Map<string, int> =
    I.Add(x, c)

The function `evalFormula` takes three arguments:
- `F` is a formula from first order logic,
- `S` is a first order structure,
- `I` is a variable assignment.

It returns the truth value of the formula `F` in the structure `S` using the variable assignment `I`.  In F#, variables in patterns should start with a lower case letter.  Therefore, the subformulas are called `g` and `h`.  A formula $\forall x: G$ is true if $G$ is true for every value $c$ of $x$ from the universe.  `Seq.forall` checks whether a predicate holds for all elements of a set.  A formula $\exists x: G$ is true if $G$ is true for some value $c$ of $x$.  `Seq.exists` checks whether a predicate holds for some element of a set.

In [25]:
let rec evalFormula (F: Formula) (S: Structure) (I: Map<string, int>) : bool =
    match F with
    | Verum          -> true
    | Falsum         -> false
    | Atom (p, args) -> evalAtomic p args S I
    | Not g          -> not (evalFormula g S I)
    | And (g, h)     -> evalFormula g S I && evalFormula h S I
    | Or  (g, h)     -> evalFormula g S I || evalFormula h S I
    | Imp (g, h)     -> evalFormula g S I <= evalFormula h S I
    | Iff (g, h)     -> evalFormula g S I =  evalFormula h S I
    | Xor (g, h)     -> evalFormula g S I <> evalFormula h S I
    | Forall (x, g)  -> Seq.forall (fun c -> evalFormula g S (modify I x c)) S.Universe
    | Exists (x, g)  -> Seq.exists (fun c -> evalFormula g S (modify I x c)) S.Universe

## Checking whether $\mathcal{S}$ is a Group

In [26]:
printfn "evalFormula %s S I = %b" G1 (evalFormula F1 S I)
printfn "evalFormula %s S I = %b" G2 (evalFormula F2 S I)
printfn "evalFormula %s S I = %b" G3 (evalFormula F3 S I)
printfn "evalFormula %s S I = %b" G4 (evalFormula F4 S I)

evalFormula 

∀x:Equals(Multiply(E(),x),x)

 S I = 

true

evalFormula 

...

true

evalFormula 

∀x:∀y:Equals(Multiply(x,y), Multiply(y,x))

 S I = 

true

This shows that the structure $\mathcal{S}$ defined above is indeed a group.  Furthermore, it is a *commutative* group.

## Another Example

Let's show that the formula $\forall x: \exists y:p(x,y) \rightarrow \exists y:\forall x:p(x,y)$ is not *universally valid*, i.e. let's show the following:
$$ \not\models \forall x: \exists y:p(x,y) \rightarrow \exists y:\forall x:p(x,y) $$

In [27]:
let G = "∀x:∃y:P(x,y)→∃y:∀x:P(x,y)"

In [28]:
let F = parse G
F

Imp (Forall ("x", Exists ("y", Atom ("P", [Var "x"; Var "y"]))), Exists ("y", Forall ("x", Atom ("P", [Var "x"; Var "y"]))))

Our aim is to construct a structure $\mathcal{S} = \langle\mathcal{U}, \mathcal{J} \rangle$  such that
$$\mathcal{S}(F) = \mathtt{false}.$$
We interpret the predicate symbol `P` as the identity relation on the set $\{0, 1\}$.

In [29]:
let pJ = set [ [0; 0]; [1; 1] ]

In [30]:
let S2 = { Universe = set [ 0; 1 ]; Functions = Map.empty; Predicates = Map [ ("P", pJ) ] }

In [31]:
let I2 = Map [ ("x", 0); ("y", 0) ]

In [32]:
evalFormula F S2 I2

false